# 4 - Bronze

Objetivo: ter acesso a fonte preservando sua estrutura original.

Apesar da extensão `.xls`, a fonte apresenta estrutura tabular delimitada por vírgulas, motivo da leitura utilizar o leitor CSV.

**4.1 - Preparando o Bronze para receber nomes válidos, sem alterar os valores dos dados:**

In [0]:
from pyspark.sql import functions as F

SOURCE_PATH = "/Workspace/Shared/MVP-Engenharia de Dados-MFarias/StudentsPerformance.xls"

BRONZE_TABLE = "main.students_performance.students_performance_bronze"

# Leitura da fonte
df_source = (
    spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", False)
        .csv(SOURCE_PATH)
)

print("Registros:", df_source.count())
print("Colunas originais:")
print(df_source.columns)

display(df_source.limit(10))

Registros: 1000
Colunas originais:
['gender', 'race/ethnicity', 'parental level of education', 'lunch', 'test preparation course', 'math score', 'reading score', 'writing score']


gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
female,group B,bachelor's degree,standard,none,72,72,74
female,group C,some college,standard,completed,69,90,88
female,group B,master's degree,standard,none,90,95,93
male,group A,associate's degree,free/reduced,none,47,57,44
male,group C,some college,standard,none,76,78,75
female,group B,associate's degree,standard,none,71,83,78
female,group B,some college,standard,completed,88,95,92
male,group B,some college,free/reduced,none,40,43,39
male,group D,high school,free/reduced,completed,64,64,67
female,group B,high school,free/reduced,none,38,60,50


**4.2 - Criação do Bronze com nomes compatíveis com Delta:**

In [0]:
df_bronze = (
    df_source
        .withColumnRenamed("race/ethnicity", "race_ethnicity")
        .withColumnRenamed("parental level of education", "parental_level_of_education")
        .withColumnRenamed("test preparation course", "test_preparation_course")
        .withColumnRenamed("math score", "math_score")
        .withColumnRenamed("reading score", "reading_score")
        .withColumnRenamed("writing score", "writing_score")
)

print("Colunas Bronze:")
print(df_bronze.columns)

display(df_bronze.limit(10))

Colunas Bronze:
['gender', 'race_ethnicity', 'parental_level_of_education', 'lunch', 'test_preparation_course', 'math_score', 'reading_score', 'writing_score']


gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
female,group B,bachelor's degree,standard,none,72,72,74
female,group C,some college,standard,completed,69,90,88
female,group B,master's degree,standard,none,90,95,93
male,group A,associate's degree,free/reduced,none,47,57,44
male,group C,some college,standard,none,76,78,75
female,group B,associate's degree,standard,none,71,83,78
female,group B,some college,standard,completed,88,95,92
male,group B,some college,free/reduced,none,40,43,39
male,group D,high school,free/reduced,completed,64,64,67
female,group B,high school,free/reduced,none,38,60,50


**4.3 - Armazenando os dados padronizados na tabela Bronze**

In [0]:
(
    df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(BRONZE_TABLE)
)

print(f"Bronze armazenado em: {BRONZE_TABLE}")


Bronze armazenado em: main.students_performance.students_performance_bronze
